# Camera-Height Effects on Wet-Road Reflection Artifacts in 3DGS

## Objective
This project investigates whether wet-road reflection artifacts in Scaniverse
3D Gaussian Splatting reconstructions change according to camera height.
Three scans of the same wet-road scene (puddle + a can) were captured at
approximately **30 cm, 60 cm, and 100 cm** — only the camera height differs.

**Hypothesis:** the lower the camera, the more below-surface reflection is reconstructed.

## Analysis pipeline
1. Load Scaniverse-exported PLY files (relative path `../data/raw/`)
2. Crop distant points (median-centered radius)
3. Estimate the road plane with RANSAC + SVD refit
4. Compute signed distance from the road plane
5. Restrict to the puddle **ROI** and count below-plane artifacts
6. Normalize artifact counts by ground points (`refl_ratio`), also report raw counts
7. Sanity-check scale/coordinates across the three scans
8. Test threshold sensitivity (REFL_LO × REFL_HI sweep)

> Large PLY files are **not** committed to the repo. Place `low.ply`, `middle.ply`,
> `high.ply` under `data/raw/` (see `data/README.md`).

*(The first `middle.ply` wet/dry surface-property cells are an optional additional
analysis; the height comparison below is the main result.)*

In [ ]:
# =========================
# CONFIG — 모든 튜닝 파라미터를 여기 모음
# 값만 바꾸면 아래 셀들은 수정할 필요 없음
# =========================
from pathlib import Path

# --- 레포 루트 자동 탐색 ---
# 노트북을 repo 루트에서 돌리든 notebooks/ 안에서 돌리든
# data/raw 가 있는 폴더를 위로 올라가며 찾는다. (경로 오류 방지)
def _find_repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw").exists():
            return base
    return Path.cwd()   # 못 찾으면 현재 폴더 (데이터 아직 안 넣은 경우)

REPO_ROOT = _find_repo_root()
DATA_DIR  = REPO_ROOT / "data" / "raw"

# --- 결과 그림 저장 폴더 (없으면 생성) ---
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT :", REPO_ROOT)
print("DATA_DIR  :", DATA_DIR, "(exists:", DATA_DIR.exists(), ")")

# --- 입력 (단일 스캔 표면 분석용) ---
PLY_PATH = DATA_DIR / "middle.ply"

# --- 포인트 크롭 (중앙값 기준 반경, m) ---
CROP_RADIUS = 5.0

# --- 평면 피팅 (RANSAC) ---
SEED            = 0
PLANE_THR       = 0.01     # inlier 거리 임계 (m)
PLANE_ITERS     = 1000     # RANSAC 반복 횟수
PLANE_SUBSAMPLE = 30000    # RANSAC용 최대 표본 수
PLANE_SEED_BAND = 0.08     # 초기 지면 후보 Y 밴드 (m)

# --- 캔 마스킹 (지면 지도에서 제외할 원기둥) ---
CAN_CENTER = (-0.32, 0.12)   # (X, Z) m — middle 기준, 이상하면 수정
CAN_RADIUS = 0.12            # m

# --- top-down 셀 지도 ---
GRID_X    = (-1.0, 1.2)     # (min, max) m
GRID_Z    = (-1.2, 1.0)     # (min, max) m
GRID_CELL = 0.04            # 셀 크기 (m)

# --- 지면 밴드 & ROI & wet/dry/paint 분류 ---
GROUND_BAND = 0.05           # |signed dist| < 이 값 = 지면 (m)
ROI_CENTER  = (0.15, -0.20)  # (X, Z) m — 단일(middle) 스캔용
ROI_RADIUS  = 1.0            # m
PAINT_PCT = 92   # luma 이 퍼센타일 이상 = 차선 도색
WET_PCT   = 30   # luma 이 퍼센타일 이하 = 젖음 후보
DRY_PCT   = 55   # luma 이 퍼센타일 이상 = 마름 후보

# --- 반경 계층 분석 ---
RADIUS_BINS   = [(0.3, 0.5), (0.5, 0.7), (0.7, 0.9)]  # (lo, hi) m
MIN_BIN_COUNT = 500   # 이보다 적으면 해당 구간 스킵

# --- SH DC 상수 (색 복원용, 여러 셀 공용) ---
C0 = 0.28209479177387814

# --- 반사(수면 아래 유령) 분석 ---
REFL_LO = 0.03    # 평면 두께 스킵: 이보다 가까우면 지면으로 봄 (m)
REFL_HI = 1.00    # 평면 아래로 얼마나 볼지 (m)

# 3개 스캔: 전부 같은 젖은 노면(물웅덩이+캔), 카메라 높이만 다름
# height(cm) -> PLY 경로
SCANS = {
    30:  DATA_DIR / "low.ply",
    60:  DATA_DIR / "middle.ply",
    100: DATA_DIR / "high.ply",
}

# --- 스캔별 물웅덩이 ROI 중심 (X, Z) m ---
# 세 스캔은 각자 독립 좌표계라 하나의 중심을 공유하면 안 됨.
# 아래 값은 '초기 추정치'. side-view / top-down 진단 셀을 보고 각각 맞춰라.
# USE_ROI=False 로 두면 ROI 없이 (기존처럼) 전체 크롭에서 계산.
USE_ROI = True
ROI_CENTERS = {
    30:  (0.15, -0.20),
    60:  (0.15, -0.20),
    100: (0.15, -0.20),
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from plyfile import PlyData

v = PlyData.read(str(PLY_PATH))["vertex"]
pts_all = np.stack([v["x"], v["y"], v["z"]], axis=1)

med = np.median(pts_all, axis=0)
keep = np.linalg.norm(pts_all - med, axis=1) < CROP_RADIUS
pts = pts_all[keep]

names = v.data.dtype.names

dc = np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], axis=1)[keep]
rgb = np.clip(0.5 + C0 * dc, 0, 1)
luma = rgb @ np.array([0.2126, 0.7152, 0.0722])

print("loaded:", PLY_PATH)
print("points:", pts.shape)
print("columns:", len(names))

In [ ]:
# =========================
# Plane fitting + signed distance 계산
# =========================
def fit_plane(P, thr=PLANE_THR, iters=PLANE_ITERS, seed=SEED,
              subsample=PLANE_SUBSAMPLE):
    rng = np.random.default_rng(seed)

    if len(P) > subsample:
        S = P[rng.choice(len(P), subsample, replace=False)]
    else:
        S = P

    best_cnt = -1
    best_n = None
    best_d = None

    for _ in range(iters):
        p = S[rng.choice(len(S), 3, replace=False)]

        n = np.cross(p[1] - p[0], p[2] - p[0])
        L = np.linalg.norm(n)

        if L < 1e-9:
            continue

        n = n / L
        d = -n @ p[0]

        cnt = (np.abs(S @ n + d) < thr).sum()

        if cnt > best_cnt:
            best_cnt = cnt
            best_n = n
            best_d = d

    # inlier로 재피팅
    inlier = np.abs(S @ best_n + best_d) < thr
    Q = S[inlier]

    c = Q.mean(axis=0)
    # full_matrices=False: U를 (N,N)이 아닌 (N,3)으로만 계산 (281s -> <0.2s)
    _, _, vh = np.linalg.svd(Q - c, full_matrices=False)
    n = vh[-1]
    d = -n @ c

    # Y-up 기준으로 지면 위가 양수
    if n[1] < 0:
        n = -n
        d = -d

    return n, d


y0 = np.median(pts[:, 1])
plane_seed = np.abs(pts[:, 1] - y0) < PLANE_SEED_BAND

n, d0 = fit_plane(pts[plane_seed])

dist = pts @ n + d0

tilt_deg = np.degrees(np.arccos(np.clip(abs(n[1]), 0, 1)))

print("normal:", n)
print(f"tilt from Y-up: {tilt_deg:.3f} deg")
print("dist median:", np.median(dist))
print("dist p1/p99:", np.percentile(dist, [1, 99]))

In [ ]:
sc = np.exp(np.stack([v[f"scale_{i}"] for i in range(3)], axis=1))[keep]
smax = sc.max(axis=1)

opacity_raw = v["opacity"][keep]
alpha = 1 / (1 + np.exp(-opacity_raw))

rest_names = [f"f_rest_{i}" for i in range(45) if f"f_rest_{i}" in names]

if len(rest_names) == 45:
    rest = np.stack([v[name] for name in rest_names], axis=1)[keep]
    rest2 = (rest ** 2).sum(axis=1)
    sigma = C0 * np.sqrt(rest2)
    contrast = sigma / np.maximum(luma, 1e-3)
else:
    print("f_rest fields not found or incomplete.")
    sigma = np.zeros(len(pts))
    contrast = np.zeros(len(pts))

print("smax median:", np.median(smax))
print("alpha median:", np.median(alpha))
print("sigma median:", np.median(sigma))

In [ ]:
xc, zc = CAN_CENTER
in_can = np.hypot(pts[:, 0] - xc, pts[:, 2] - zc) < CAN_RADIUS

bx = np.arange(GRID_X[0], GRID_X[1], GRID_CELL)
bz = np.arange(GRID_Z[0], GRID_Z[1], GRID_CELL)

def cellmap(w=None, band=GROUND_BAND):
    m = (np.abs(dist) < band) & ~in_can

    c, _, _ = np.histogram2d(
        pts[m, 0],
        pts[m, 2],
        bins=[bx, bz]
    )

    if w is None:
        return np.where(c > 0, c, np.nan)

    s, _, _ = np.histogram2d(
        pts[m, 0],
        pts[m, 2],
        bins=[bx, bz],
        weights=w[m]
    )

    # c==0 셀은 그대로 NaN (0으로 나눠 RuntimeWarning 내지 않음)
    result = np.full_like(s, np.nan, dtype=float)
    np.divide(s, c, out=result, where=c > 0)
    return result

Lmap = cellmap(luma)
Dmap = cellmap(None)
Smaxmap = cellmap(smax)
Alphamap = cellmap(alpha)
Sigmamap = cellmap(sigma)
Contrastmap = cellmap(contrast)

maps = [Lmap, Dmap, Smaxmap, Alphamap, Sigmamap, Contrastmap]
titles = ["luma", "density", "smax", "alpha", "sigma_view", "SH contrast"]

fig, ax = plt.subplots(2, 3, figsize=(16, 10))

for a, M, t in zip(ax.ravel(), maps, titles):
    im = a.imshow(
        M.T,
        origin="lower",
        extent=[bx[0], bx[-1], bz[0], bz[-1]],
        aspect="equal"
    )
    a.set_title(t)
    a.set_xlabel("X (m)")
    a.set_ylabel("Z (m)")
    plt.colorbar(im, ax=a)

plt.tight_layout()
plt.show()

In [ ]:
gnd = (np.abs(dist) < GROUND_BAND) & ~in_can

rx, rz = ROI_CENTER
roi = gnd & (np.hypot(pts[:, 0] - rx, pts[:, 2] - rz) < ROI_RADIUS)

paint = roi & (luma > np.percentile(luma[roi], PAINT_PCT))
wet = roi & ~paint & (luma < np.percentile(luma[roi], WET_PCT))
dry = roi & ~paint & (luma > np.percentile(luma[roi], DRY_PCT))

print("===== POINT COUNTS =====")
print("roi:", roi.sum())
print("wet:", wet.sum())
print("dry:", dry.sum())
print("paint:", paint.sum())

print("\n===== MEDIAN METRICS =====")
for m, nm in [(wet, "wet"), (dry, "dry"), (paint, "paint")]:
    print(
        nm,
        "luma", np.median(luma[m]),
        "smax", np.median(smax[m]),
        "alpha", np.median(alpha[m]),
        "sigma", np.median(sigma[m]),
        "contrast", np.median(contrast[m]),
    )

print("\n===== WET - DRY =====")
print("Delta luma:", np.median(luma[wet]) - np.median(luma[dry]))
print("Delta smax:", np.median(smax[wet]) - np.median(smax[dry]))
print("Delta alpha:", np.median(alpha[wet]) - np.median(alpha[dry]))
print("Delta sigma:", np.median(sigma[wet]) - np.median(sigma[dry]))

r = np.linalg.norm(pts[:, [0, 2]] - med[[0, 2]], axis=1)

print("\n===== RADIUS STRATIFIED =====")
for lo, hi_r in RADIUS_BINS:
    b = (r >= lo) & (r < hi_r)

    if (wet & b).sum() < MIN_BIN_COUNT or (dry & b).sum() < MIN_BIN_COUNT:
        print(f"r={lo}-{hi_r}: not enough points")
        continue

    print(
        f"r={lo}-{hi_r}",
        f"Nwet={(wet & b).sum()}",
        f"Ndry={(dry & b).sum()}",
        f"smax {np.median(smax[wet & b]):.5f} / {np.median(smax[dry & b]):.5f}",
        f"alpha {np.median(alpha[wet & b]):.3f} / {np.median(alpha[dry & b]):.3f}",
        f"sigma {np.median(sigma[wet & b]):.4f} / {np.median(sigma[dry & b]):.4f}",
    )

# 반사 포인트 vs 카메라 높이 분석

**세팅:** 3개 스캔 모두 같은 젖은 노면(물웅덩이 + 옆에 캔). 카메라 높이만 30/60/100cm.

**가설:** 카메라가 낮을수록 물에 비친 반사가 많이 재구성된다.

**지표:** 수면 평면 아래에 생긴 유령 포인트 수를 지면 포인트로 정규화한 **below-plane 비율**(`refl_ratio`). 물웅덩이 **ROI 안에서만** 계산해 배경 물체·재구성 노이즈를 배제한다. 비율뿐 아니라 **원시 개수(`n_refl`)**도 함께 확인해 분모 효과가 아님을 본다.

> **주의 — 먼저 검증할 것:** 세 스캔은 각자 독립 좌표계다. 아래 sanity-check 셀에서 (1) 세 스캔의 축 방향/스케일(extent)이 같은지, (2) 아래쪽 유령이 물웅덩이 자리에 모이는지, (3) `ROI_CENTERS`가 실제 물웅덩이를 덮는지 **눈으로 확인**하고, 필요하면 CONFIG의 `ROI_CENTERS`를 스캔별로 맞춘 뒤 결과 셀을 다시 실행할 것.

위쪽 셀들(map/wet-dry)은 단일 스캔 표면 분석용 — 실행 안 해도 됨. 단, **셀 0(CONFIG)과 `fit_plane` 정의 셀은 먼저 한 번 실행**해 둘 것.

In [ ]:
# =========================
# 스캔 1개 -> 크롭 -> 지면 평면 -> signed distance
# (기존 로딩/평면 로직을 함수로 감쌈. fit_plane 은 위 셀 정의를 그대로 사용)
# =========================
def load_scan(path):
    v = PlyData.read(str(path))["vertex"]
    P = np.stack([v["x"], v["y"], v["z"]], axis=1)

    med = np.median(P, axis=0)
    keep = np.linalg.norm(P - med, axis=1) < CROP_RADIUS
    pts = P[keep]

    dc = np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], axis=1)[keep]
    rgb = np.clip(0.5 + C0 * dc, 0, 1)
    luma = rgb @ np.array([0.2126, 0.7152, 0.0722])

    seed = np.abs(pts[:, 1] - np.median(pts[:, 1])) < PLANE_SEED_BAND
    n, d0 = fit_plane(pts[seed])
    dist = pts @ n + d0
    return pts, luma, dist, med


def roi_mask(pts, height):
    """USE_ROI=True 면 스캔별 물웅덩이 중심 기준 원형 ROI, 아니면 전체 True."""
    if not USE_ROI:
        return np.ones(len(pts), dtype=bool)
    rx, rz = ROI_CENTERS[height]
    return np.hypot(pts[:, 0] - rx, pts[:, 2] - rz) < ROI_RADIUS


def reflection_metrics(path, height, refl_lo=None, refl_hi=None, ground_band=None):
    # 인자 없으면 CONFIG 기본값 사용
    refl_lo     = REFL_LO      if refl_lo     is None else refl_lo
    refl_hi     = REFL_HI      if refl_hi     is None else refl_hi
    ground_band = GROUND_BAND  if ground_band is None else ground_band

    pts, luma, dist, med = load_scan(path)

    in_roi = roi_mask(pts, height)                              # 물웅덩이 주변으로 제한
    gnd  = (np.abs(dist) < ground_band) & in_roi               # 진짜 지면 밴드
    refl = (dist < -refl_lo) & (dist > -refl_hi) & in_roi      # 수면 "아래" 유령 밴드

    # depth: ROI 안에서만 -> 배경 물체/이상치 영향 배제
    roi_dist = dist[in_roi]
    depth_p01 = float(np.percentile(roi_dist, 1)) if roi_dist.size else np.nan

    # 반사 후보만의 깊이 중앙값 (해석 쉬움: 평면 아래 몇 m에 유령이 앉나)
    refl_depth_med = float(np.median(-dist[refl])) if refl.sum() else np.nan

    return {
        "n_ground":       int(gnd.sum()),
        "n_refl":         int(refl.sum()),
        "refl_ratio":     refl.sum() / max(gnd.sum(), 1),  # 지면 대비 반사 비율
        "depth_p01":      depth_p01,                        # ROI 내부 아래쪽 1%
        "refl_depth_med": refl_depth_med,                   # 반사 후보 깊이 중앙값
    }

In [ ]:
# =========================
# 높이별 추이: 반사 비율 + 반사 깊이
# 낮은 높이에서 값이 크고 높이 올라갈수록 감소 -> 가설 지지
# =========================
heights = [m["height"] for m in rows]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].plot(heights, [m["refl_ratio"] for m in rows],
           marker="o", lw=2, color="tab:blue")
ax[0].set_title("below-plane reflection ratio")
ax[0].set_ylabel("refl / ground")

ax[1].plot(heights, [m["depth_p01"] for m in rows],
           marker="^", lw=2, color="tab:green")
ax[1].set_title("depth p01 within ROI (how far below plane)")
ax[1].set_ylabel("signed dist (m)")

for a in ax:
    a.set_xlabel("camera height (cm)")
    a.set_xticks(heights)
    a.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "reflection_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =========================
# sanity check (세 스캔 동시):
#   왼쪽  = 측면도 (Z vs 평면까지 거리). 유령이 평면 "아래"(빨간선 아래)에 앉는지.
#   오른쪽 = 위에서 본 그림 (X vs Z, 지면 밴드). 물웅덩이 위치 + ROI 원 확인.
# 목적:
#   1) 세 스캔의 축 방향/스케일이 같은가 (extent 프린트로도 확인)
#   2) 아래쪽 유령이 물웅덩이 자리에 모여 있는가
#   3) ROI_CENTERS 값이 실제 물웅덩이를 덮는가  -> 아니면 CONFIG에서 수정
# =========================
import matplotlib.patches as mpatches

fig, ax = plt.subplots(len(SCANS), 2, figsize=(13, 4.2 * len(SCANS)))
if len(SCANS) == 1:
    ax = ax[None, :]

for row, h in enumerate(sorted(SCANS)):
    try:
        pts, luma, dist, med = load_scan(SCANS[h])
    except FileNotFoundError:
        ax[row, 0].set_title(f"{h}cm: 파일 없음")
        continue

    # --- 스케일/좌표 비교용 extent ---
    ext = pts.max(axis=0) - pts.min(axis=0)
    print(f"{h:>4}cm  N={len(pts):>7}  "
          f"extent XYZ = ({ext[0]:.2f}, {ext[1]:.2f}, {ext[2]:.2f}) m  "
          f"median XZ = ({med[0]:.2f}, {med[2]:.2f})")

    # --- 왼쪽: 측면도 (median 근처 얇은 X 슬라이스) ---
    sel = np.abs(pts[:, 0] - med[0]) < 0.3
    a0 = ax[row, 0]
    sc_ = a0.scatter(pts[sel, 2], dist[sel], c=luma[sel], s=2, cmap="viridis")
    a0.axhline(0, color="r", lw=1, label="ground plane")
    a0.axhline(-REFL_LO, color="orange", ls="--", lw=1)
    a0.axhline(-REFL_HI, color="orange", ls="--", lw=1, label="reflection band")
    a0.set_xlabel("Z (m)")
    a0.set_ylabel("signed dist (m) [+up/-down]")
    a0.set_title(f"side view @ {h}cm")
    a0.legend(fontsize=8)
    plt.colorbar(sc_, ax=a0, label="luma")

    # --- 오른쪽: 위에서 본 지면 밴드 + ROI 원 ---
    gnd = np.abs(dist) < GROUND_BAND
    a1 = ax[row, 1]
    a1.scatter(pts[gnd, 0], pts[gnd, 2], c=luma[gnd], s=2, cmap="viridis")
    rx, rz = ROI_CENTERS[h]
    a1.add_patch(mpatches.Circle((rx, rz), ROI_RADIUS, fill=False,
                                 color="red", lw=1.5, label="ROI"))
    a1.plot(rx, rz, "r+", ms=12)
    a1.set_aspect("equal")
    a1.set_xlabel("X (m)")
    a1.set_ylabel("Z (m)")
    a1.set_title(f"top-down ground @ {h}cm  ROI={ROI_CENTERS[h]}")
    a1.legend(fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "side_view_sanity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =========================
# sanity check (세 스캔 동시):
#   왼쪽  = 측면도 (Z vs 평면까지 거리). 유령이 평면 "아래"(빨간선 아래)에 앉는지.
#   오른쪽 = 위에서 본 그림 (X vs Z, 지면 밴드). 물웅덩이 위치 + ROI 원 확인.
# 목적:
#   1) 세 스캔의 축 방향/스케일이 같은가 (extent 프린트로도 확인)
#   2) 아래쪽 유령이 물웅덩이 자리에 모여 있는가
#   3) ROI_CENTERS 값이 실제 물웅덩이를 덮는가  -> 아니면 CONFIG에서 수정
# =========================
import matplotlib.patches as mpatches

fig, ax = plt.subplots(len(SCANS), 2, figsize=(13, 4.2 * len(SCANS)))
if len(SCANS) == 1:
    ax = ax[None, :]

for row, h in enumerate(sorted(SCANS)):
    try:
        pts, luma, dist, med = load_scan(SCANS[h])
    except FileNotFoundError:
        ax[row, 0].set_title(f"{h}cm: 파일 없음")
        continue

    # --- 스케일/좌표 비교용 extent ---
    ext = pts.max(axis=0) - pts.min(axis=0)
    print(f"{h:>4}cm  N={len(pts):>7}  "
          f"extent XYZ = ({ext[0]:.2f}, {ext[1]:.2f}, {ext[2]:.2f}) m  "
          f"median XZ = ({med[0]:.2f}, {med[2]:.2f})")

    # --- 왼쪽: 측면도 (median 근처 얇은 X 슬라이스) ---
    sel = np.abs(pts[:, 0] - med[0]) < 0.3
    a0 = ax[row, 0]
    sc_ = a0.scatter(pts[sel, 2], dist[sel], c=luma[sel], s=2, cmap="viridis")
    a0.axhline(0, color="r", lw=1, label="ground plane")
    a0.axhline(-REFL_LO, color="orange", ls="--", lw=1)
    a0.axhline(-REFL_HI, color="orange", ls="--", lw=1, label="reflection band")
    a0.set_xlabel("Z (m)")
    a0.set_ylabel("signed dist (m) [+up/-down]")
    a0.set_title(f"side view @ {h}cm")
    a0.legend(fontsize=8)
    plt.colorbar(sc_, ax=a0, label="luma")

    # --- 오른쪽: 위에서 본 지면 밴드 + ROI 원 ---
    gnd = np.abs(dist) < GROUND_BAND
    a1 = ax[row, 1]
    a1.scatter(pts[gnd, 0], pts[gnd, 2], c=luma[gnd], s=2, cmap="viridis")
    rx, rz = ROI_CENTERS[h]
    a1.add_patch(mpatches.Circle((rx, rz), ROI_RADIUS, fill=False,
                                 color="red", lw=1.5, label="ROI"))
    a1.plot(rx, rz, "r+", ms=12)
    a1.set_aspect("equal")
    a1.set_xlabel("X (m)")
    a1.set_ylabel("Z (m)")
    a1.set_title(f"top-down ground @ {h}cm  ROI={ROI_CENTERS[h]}")
    a1.legend(fontsize=8)

plt.tight_layout()
plt.show()

# 파라미터 민감도 스윕

`REFL_LO`(평면 근처 스킵 두께)와 `REFL_HI`(아래로 보는 깊이)를 여러 값으로 바꿔가며 refl_ratio를 다시 계산한다.

**보이려는 것:** 임계값을 어떻게 잡아도 `30cm > 60cm > 100cm` 순서(낮은 카메라일수록 반사 많음)가 뒤집히지 않는다 → 결과가 특정 튜닝값의 산물이 아니라 견고하다.

In [ ]:
# =========================
# 스윕 결과 시각화: 모든 (lo,hi) 조합을 겹쳐 그림
# 선들이 서로 안 교차하고 30cm가 항상 위 -> 순서 견고
# =========================
plt.figure(figsize=(8, 5))
for (lo, hi), rr in sweep.items():
    ys = [rr[h] for h in heights]
    plt.plot(heights, ys, marker="o", alpha=0.5, lw=1,
             label=f"lo={lo}, hi={hi}")

plt.xlabel("camera height (cm)")
plt.ylabel("reflection ratio")
plt.title("Sensitivity sweep: refl_ratio vs height for all (LO, HI)")
plt.xticks(heights)
plt.grid(alpha=0.3)
plt.legend(fontsize=7, ncol=2, loc="upper right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "threshold_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

# 정규화 버전: 각 조합을 30cm=1.0 으로 맞춰 '모양'만 비교
plt.figure(figsize=(8, 5))
for (lo, hi), rr in sweep.items():
    base = rr[heights[0]]
    if np.isnan(base) or base == 0:
        continue
    ys = [rr[h] / base for h in heights]
    plt.plot(heights, ys, marker="o", alpha=0.5, lw=1)

plt.axhline(1.0, color="gray", ls="--", lw=1)
plt.xlabel("camera height (cm)")
plt.ylabel("refl_ratio (normalized to 30cm = 1.0)")
plt.title("Normalized: all curves decline together regardless of thresholds")
plt.xticks(heights)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "threshold_sensitivity_normalized.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =========================
# 스윕 결과 시각화: 모든 (lo,hi) 조합을 겹쳐 그림
# 선들이 서로 안 교차하고 30cm가 항상 위 -> 순서 견고
# =========================
plt.figure(figsize=(8, 5))
for (lo, hi), rr in sweep.items():
    ys = [rr[h] for h in heights]
    plt.plot(heights, ys, marker="o", alpha=0.5, lw=1,
             label=f"lo={lo}, hi={hi}")

plt.xlabel("camera height (cm)")
plt.ylabel("reflection ratio")
plt.title("Sensitivity sweep: refl_ratio vs height for all (LO, HI)")
plt.xticks(heights)
plt.grid(alpha=0.3)
plt.legend(fontsize=7, ncol=2, loc="upper right")
plt.tight_layout()
plt.show()

# 정규화 버전: 각 조합을 30cm=1.0 으로 맞춰 '모양'만 비교
plt.figure(figsize=(8, 5))
for (lo, hi), rr in sweep.items():
    base = rr[heights[0]]
    if np.isnan(base) or base == 0:
        continue
    ys = [rr[h] / base for h in heights]
    plt.plot(heights, ys, marker="o", alpha=0.5, lw=1)

plt.axhline(1.0, color="gray", ls="--", lw=1)
plt.xlabel("camera height (cm)")
plt.ylabel("refl_ratio (normalized to 30cm = 1.0)")
plt.title("Normalized: all curves decline together regardless of thresholds")
plt.xticks(heights)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()